# 2.4 The same trap, without a hidden variable

[02.3](02.3-simpsons_paradox.ipynb) needed a lurking grouping variable — department — for the
aggregate and the subgroups to disagree. This one needs nothing but a careless choice of
*what to count*, which makes it both easier to fall into and easier to check for: there is no
third variable to go looking for, only a question — "what is one unit, here?" — that a claim
about people quietly skips past.

Back to the IRC corpus from lesson 1, and to lesson 1's pipeline, now an import rather than a
block of regex to paste.

In [ ]:
import pandas as pd
from goad_toolkit.visualizer import (
    Annotate,
    BasePlot,
    PlotSettings,
    ScatterPlot,
)

from scripts.pipelines import build_irc_pipeline
from scripts.plots import BarPlot
from wa_analyzer.data import load_showcase

## 2.4.1 The data

The parse, the calendar columns and the three regex features all rerun here exactly as they
did in lesson 1, including the coverage line it logs.

In [ ]:
msgs = build_irc_pipeline().apply(load_showcase("ubuntu_irc"))
msgs["length"] = msgs.message.str.len()
print(f"{len(msgs):,} messages, {msgs.author.nunique():,} authors")

**The question: do people in one channel write longer messages than people in the other?**

Note the wording. It is a claim about *people*. Keep that in mind, and count messages
first — because that is what almost everyone does first, including you a paragraph from now.

In [ ]:
by_message = msgs.groupby("channel").length.agg(["size", "mean", "sem"])
by_message["ci95"] = 1.96 * by_message["sem"]

per_author = msgs.groupby(["channel", "author"]).length.agg(["size", "mean"])
per_author = per_author[per_author["size"] >= 30].reset_index()

by_author = per_author.groupby("channel")["mean"].agg(["size", "mean", "sem"])
by_author["ci95"] = 1.96 * by_author["sem"]

## 2.4.2 What an errorbar is, and why it belongs here

A mean computed from a sample is not the true mean — it is an estimate, and a different
sample from the same population would give a slightly different number. A 95% confidence
interval is a range built so that, under ordinary assumptions, it would contain the true mean
about 95 times out of 100 if you repeated the sampling. Plotted as an errorbar, it turns "the
mean is 87.3" into "the mean is 87.3, and anywhere from about 85 to 90 is consistent with this
data" — which is the honest version of the claim.

Two bars with no errorbars invite you to compare two numbers as if they were exact. Two bars
with errorbars that overlap are telling you, visually, that the difference between them might
be sampling noise rather than a real one. And the width of the interval is not fixed — it
shrinks as the number of *independent units* behind the mean grows, which is exactly the
count-messages-versus-count-people question this notebook is about, made visible on the
plot itself rather than argued about in prose.

`GroupedBarPlot` cannot draw these errorbars: seaborn derives its own from the raw
observations it is handed, and `by_message`/`by_author` are already-summarised means with
intervals computed separately. A small `BasePlot` subclass draws the bars, then places one
errorbar per bar by looking it up on `(category, hue level)` — seaborn draws one container
per hue level in `hue_order` and the categories along the x-axis in tick order, so those two
are what the lookup keys on, and the check below turns a layout change into a failure rather
than a silently misplaced error bar.

In [ ]:
UNITS = ["counting messages", "counting people"]

compare = pd.concat([
    by_message.assign(unit=UNITS[0]),
    by_author.assign(unit=UNITS[1]),
]).reset_index()[["channel", "unit", "mean", "ci95"]]


class BarPlotWithError(BasePlot):
    """A grouped bar chart carrying intervals that were computed elsewhere."""

    def build(self, data: pd.DataFrame, x: str, y: str, hue: str, error: str,
              hue_order: list[str], **kwargs):
        import seaborn as sns

        sns.barplot(data=data, x=x, y=y, hue=hue, hue_order=hue_order,
                    ax=self.ax, **kwargs)
        if self.ax is None:
            raise ValueError("create_figure() must run before build()")

        intervals = data.set_index([x, hue])[error]
        categories = [label.get_text() for label in self.ax.get_xticklabels()]
        if len(self.ax.containers) != len(hue_order):
            raise ValueError(
                f"seaborn drew {len(self.ax.containers)} containers for "
                f"{len(hue_order)} hue levels; the error bars would be misplaced"
            )

        for container, level in zip(self.ax.containers, hue_order):
            for patch, category in zip(container, categories):
                self.ax.errorbar(
                    patch.get_x() + patch.get_width() / 2,
                    patch.get_height(),
                    yerr=intervals[(category, level)],
                    fmt="none", ecolor="black", capsize=4,
                )

        # Left to itself matplotlib puts this over the bars, which is the exact
        # thing 02.2's "grey first, then one colour" is about.
        self.ax.legend(loc="lower right", framealpha=1)
        return self.fig, self.ax


units = PlotSettings(
    figsize=(8, 4),
    title="Same question, two units of analysis",
    xlabel="",
    ylabel="mean message length (characters)",
)
fig, ax = BarPlotWithError(units).plot(
    data=compare, x="channel", y="mean", hue="unit", error="ci95",
    hue_order=UNITS, palette=["#cccccc", "#c44e52"],
)

The answer **changes sign**. Counting messages, `#ubuntu-uk` is shorter. Counting people,
`#ubuntu-uk` is longer — and the errorbars on both units are tight enough that this is not
noise in either direction. Both numbers are correct. They answer different questions:

- *counting messages* — "if I pick a random message from this channel, how long is it?"
- *counting people* — "if I pick a random regular of this channel, how long are their
  messages?"

The claim we made was about people. Only one of these tests it.

### Why it flips

In [ ]:
uk_channel = "#ubuntu-uk"

top8 = (
    msgs[msgs.channel == uk_channel].author.value_counts().head(8)
    .rename_axis("author").reset_index(name="n")
)
top8_share = top8.n.sum() / (msgs.channel == uk_channel).sum()

uk = per_author[per_author.channel == uk_channel]
most_verbose = uk.nlargest(1, "mean").iloc[0]

flip = PlotSettings(
    figsize=(13, 4),
    title="Why the answer flips",
    subplot_titles=[
        f"the {len(top8)} busiest authors write {top8_share:.0%} of #ubuntu-uk's messages",
        "...and the busiest are not the most verbose",
    ],
    xlabel="",
    ylabel="",
)

host = BarPlot(flip)  # any BasePlot subclass hosts create_figure/plot_on_axes
fig, axes = host.create_figure(n_plots=2)

host.plot_on_axes(BarPlot(flip), axes[0], data=top8, x="author", y="n", color="#cccccc")
axes[0].tick_params(axis="x", rotation=45)
axes[0].set_ylabel("messages written")

host.plot_on_axes(ScatterPlot(flip), axes[1], data=uk, x="size", y="mean",
                  color="#4c72b0", alpha=0.6)
axes[1].set_xlabel("messages written")
axes[1].set_ylabel("mean message length (characters)")
host.plot_on_axes(
    Annotate(flip), axes[1],
    text=f"{most_verbose.author}: {int(most_verbose['size'])} messages, "
         f"{most_verbose['mean']:.0f} chars each",
    xy=(most_verbose["size"], most_verbose["mean"]),
    xytext=(most_verbose["size"] + uk["size"].max() * 0.15,
           most_verbose["mean"] - uk["mean"].max() * 0.15),
)
fig.tight_layout()

Two things pull in opposite directions. The message-level mean is dominated by the handful
of regulars in the left panel — 44% of everything from eight people — and they are terse. The
author-level mean gives one vote each to accounts like the one annotated on the right: a
few dozen messages, thoroughly verbose, invisible to a message-level average and decisive to
a person-level one.

That is the part worth carrying: **the fewer units you have, the more one bad unit moves the
answer.** It is the same idea as the errorbar in 2.4.2 shrinking with more independent units,
seen from the other side — cleaning matters more, not less, when you aggregate up to people.

Neither unit is the right one in general. The right one is the one your sentence is about.

> **Ask this every time you compare groups:**
>
> 1. What is my claim about — messages, people, days, conversations?
> 2. Is that the thing I counted?
> 3. How many of them are there? Not rows. Units.

### And this class doesn't belong in a notebook cell either

Same reasoning as `BarPlot` in [02.2](02.2-comparing_categories.ipynb): `BarPlotWithError`
works, and something else is going to want it — so it moves to `scripts/plots.py` alongside
`BarPlot`, unchanged.

In [ ]:
from scripts.plots import BarPlotWithError as ImportedBarPlotWithError

# The same class, from the file rather than from the cell above — the figure is identical.
fig, ax = ImportedBarPlotWithError(units).plot(
    data=compare, x="channel", y="mean", hue="unit", error="ci95",
    hue_order=UNITS, palette=["#cccccc", "#c44e52"],
)

---

**Where this goes next.** [02.5-your_turn](02.5-your_turn.ipynb) is your own chat, with the
`BarPlot` and `BarPlotWithError` built here preloaded and nothing else decided for you.